# 0. Verificar el entorno de Windows
Ejecuta este cuaderno antes de descargar datos. Comprueba el kernel, la compilación CUDA, una operación FP16 y el cálculo de las métricas por píxel con un ejemplo de resultado conocido.

In [ ]:
from pathlib import Path
from IPython import get_ipython

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)


def load_notebook(name):
    path = ROOT / "notebooks" / "shared" / (name + ".ipynb")
    get_ipython().run_line_magic("run", f'-i "{path.as_posix()}"')


load_notebook("workspace")

In [ ]:
import platform
import sys
import subprocess
import torch
from importlib.metadata import version
from tempfile import TemporaryDirectory

if (
    platform.system() != "Windows"
    or sys.version_info[:2] != (3, 11)
    or Path(sys.prefix).resolve() != (ROOT / ".venv").resolve()
):
    raise RuntimeError(
        "Abre Jupyter con uv desde el repositorio y selecciona su kernel de Python 3.11."
    )
if (
    version("anomalib") != "2.6.2"
    or torch.__version__ != "2.6.0+cu124"
    or not torch.cuda.is_available()
):
    raise RuntimeError("Comprueba uv sync --locked y el controlador NVIDIA.")
print(
    subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
        text=True,
    )
)
matrix = torch.eye(16, device="cuda", dtype=torch.float16)
assert torch.equal(matrix @ matrix, matrix)
load_notebook("experiment")
load_notebook("metrics")
with TemporaryDirectory() as temporary:
    folder = Path(temporary)
    mask = np.eye(4, dtype=bool)
    np.savez(folder / "000000.npz", score=mask.astype(float), mask=mask)
    measured = pixel_metrics(folder)
    for key in ("pixel_auroc", "pixel_ap", "au_pro"):
        assert np.isclose(measured[key], 1), key
print("Entorno CUDA y métricas verificados.")